In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import bosperrus
from bosperrus.centrality_measures import compute_centrality_measures
from datetime import datetime

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
import pytz
from timezonefinder import TimezoneFinder

In [51]:
def get_utc_offset(tz_name):
    tz = pytz.timezone(tz_name)
    return tz.utcoffset(datetime.utcnow()).total_seconds() / 3600

In [3]:
edge_list = pd.read_csv("../SNAP_data/loc-brightkite_edges.txt", delimiter="\t", header=None).values

In [4]:
try: 
    median_locations = pd.read_csv("./geo_data/median_location_per_user_brightkite.csv")
except:
    def approx_median_location(group):
        coords = group[['latitude', 'longitude']].to_numpy()
        centroid = coords.mean(axis=0)
        dists = np.sum((coords - centroid) ** 2, axis=1)  # squared distance is enough
        return group.iloc[dists.argmin()]
    logs = pd.read_csv("../SNAP_data/loc-brightkite_totalCheckins.txt", delimiter="\t", header=None)
    logs.columns = ["user", "check-in time", "latitude", "longitude", "location id"]
    median_locations = logs.groupby('user', group_keys=False).apply(approx_median_location).drop("user", axis=1)
    median_locations = median_locations.drop(["check-in time", "location id"], axis=1)
    median_locations.to_csv("./geo_data/median_location_per_user_brightkite.csv")

In [6]:
cities = pd.read_csv("./geo_data/cities_us.csv").drop("Unnamed: 0", axis=1)

In [7]:
gdf_cities = gpd.GeoDataFrame(
    cities,
    geometry=gpd.points_from_xy(cities.longitude, cities.latitude),
    crs="EPSG:4326"
)

In [59]:
def get_utc_offset(tz_name):
    try:
        tz = pytz.timezone(tz_name)
        dt = datetime.now(tz)  # ✅ localized
        return dt.utcoffset().total_seconds() / 3600
    except:
        return None
        
gdf_users = gpd.GeoDataFrame(
    median_locations,
    geometry=gpd.points_from_xy(median_locations.longitude, median_locations.latitude),
    crs="EPSG:4326"
)
gdf_users["timezone"] = [
    tf.timezone_at(lng=x, lat=y)
    for x, y in zip(gdf_users.geometry.x, gdf_users.geometry.y)
]
#usa = ox.geocode_to_gdf("United States") 
#gdf_usa = gpd.sjoin(gdf_users, usa, predicate="within")
#minx, miny, maxx, maxy = -125, 24, -66, 50
#gdf_usa = gdf_usa.cx[minx:maxx, miny:maxy]
gdf_users["utc_offset"] = gdf_users["timezone"].apply(get_utc_offset)
ny_tz = pytz.timezone("America/New_York")
ny_offset = ny_tz.utcoffset(datetime.utcnow()).total_seconds() / 3600
gdf_users["offset_vs_ny"] = gdf_users["utc_offset"] - ny_offset

In [ ]:
measures = ["closeness", "degree", "clustering"]
# Node IDs in the SNAP edge list are 0-indexed; N = max node ID + 1
N = int(edge_list.max()) + 1
scores = compute_centrality_measures(edge_list, N=N, measures=measures)
distances = pd.Series(np.abs(gdf_users["offset_vs_ny"]).values, name="offset_vs_ny")
bf = bosperrus.Flow.from_distances_and_scores(distances=distances, scores=scores)
bf.flow(measures=measures)

In [ ]:
df = bf.observations.dropna()

In [ ]:
plt.scatter(np.abs(df["offset_vs_ny"]), df["degree"])
plt.scatter(bf.observations["offset_vs_ny"], bf.best_fits['degree'].S_model)

In [ ]:
# bf.flow() was already called during Flow setup above

In [90]:
bf.best_fits['degree'].observed_effect_strength

np.float64(0.12617768900592644)

In [87]:
bf.best_fits['degree'].params

{'piecewise_linear_b': np.float64(5.9999975409886055),
 'piecewise_linear_m': np.float64(-0.6776508903287425),
 'piecewise_linear_c': np.float64(18.14476886204538)}

In [89]:
bf.fit_quality

,measure,best_fit_type,entropy_AIC_weights,observed_half_life,observed_effect_strength,included samples,affected samples,piecewise_linear_b,piecewise_linear_m,piecewise_linear_c,Piecewise Linear Fit_scaled_relative_likelihood_over_Constant Fit,Piecewise Linear Fit_AIC_weight,Exponential Saturation Fit_scaled_relative_likelihood_over_Constant Fit,Exponential Saturation Fit_AIC_weight,Michaelis-Menten Fit_scaled_relative_likelihood_over_Constant Fit,Michaelis-Menten Fit_AIC_weight,constant_c
0,closeness,Piecewise Linear Fit,1.098612,3.000005,0.006329,0.88284,0.855639,6.000009,-0.000469,0.223890,1.000042,0.333343,0.999998,0.333329,0.999995,0.333328,NaN
1,degree,Piecewise Linear Fit,1.098612,2.999999,0.126178,0.88284,0.724351,5.999998,-0.677651,18.144769,1.000645,0.333388,1.000406,0.333308,1.000395,0.333304,NaN
2,clustering,Constant Fit,1.098612,0.000000,0.000000,0.88284,0.000000,NaN,NaN,NaN,1.000000,0.333337,0.999983,0.333332,0.999979,0.333331,0.147956
